# E3 · Más diversidad de modelos base para el stacking
**Parte de:** E0 (stacking de 5 modelos).

**Hipótesis.** El stacking sumó +0,6 puntos al combinar cinco modelos lineales casi iguales. Si se agregan modelos que **se equivocan en tweets distintos**, por ver el texto de otra forma o por no ser lineales, el metamodelo tendría más información para combinar.

Modelos nuevos (código en `toxic_base.py`, sección E3):

| Modelo | Qué cambia |
|---|---|
| **LR-charc** | LR con n-gramas de caracteres `char` 2-6 que **cruzan palabras** (`xuser callate p`), en vez de `char_wb` |
| **NB-LR-charc** | NB-LR con esa misma vista de caracteres |
| **LGBM** | Árboles potenciados (LightGBM) sobre el TF-IDF reducido a 300 dimensiones con SVD + las 38 variables de léxico sin escalar. Es el único modelo **no lineal** |
| **Cadena** | Cadena de clasificadores HS → AG → TR: cada etiqueta usa las anteriores como variable |

### Glosario de esta prueba
| Término | Significado |
|---|---|
| **HS / TR / AG** | Las 3 etiquetas: discurso de odio / odio dirigido a una persona concreta / agresividad |
| **F1 medio** | F1 de la clase 1 de cada etiqueta, promediado entre HS, TR y AG (métrica principal) |
| **CV repetida** | Validación cruzada de 5 particiones repetida con 3 semillas (42, 43, 44); se reporta media ± desviación |
| **OOF** | *Out-of-fold*: predicción de cada fila del train hecha por un modelo que no la vio al entrenar |
| **Stacking** | Un metamodelo (regresión logística) aprende a combinar las probabilidades OOF de los modelos base |
| **`char` vs `char_wb`** | `char_wb` corta los n-gramas de caracteres en los bordes de palabra; `char` los deja cruzar espacios |
| **SVD** | Reducción de dimensión: resume miles de columnas TF-IDF en 300 |
| **LightGBM** | Modelo de árboles de decisión potenciados (*gradient boosting*), no lineal |
| **Cadena de clasificadores** | Predice HS, usa esa predicción para AG, y ambas para TR |

El glosario completo está en `RESUMEN.md`.

In [1]:
import os, sys, platform
sys.path.insert(0, os.path.abspath('.'))          # el notebook vive en experimentos/, junto a toxic_base.py
import numpy as np, pandas as pd, sklearn, scipy
import toxic_base as tb
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 120)
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__,
      '| scikit-learn', sklearn.__version__, '| scipy', scipy.__version__)
print('Semillas de la CV repetida:', tb.SEEDS, '| particiones:', tb.N_FOLDS)
d = tb.Datos()
print('Datos en', d.data_dir, '| train', d.train.shape, '| test', d.test.shape)

Python 3.12.3 | numpy 2.5.4 | pandas 3.0.6 | scikit-learn 1.9.1 | scipy 1.18.1
Semillas de la CV repetida: (42, 43, 44) | particiones: 5


Datos en /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/tweet-toxic-2026-puj | train (1731, 5) | test (1154, 2)


## 1. Modelos nuevos por separado

In [2]:
NUEVOS = ['LR-charc', 'NB-LR-charc', 'LGBM', 'Cadena']
R = {}
for b in tb.BASES_ORIGINALES + NUEVOS:
    R[b] = tb.evaluar(d, {s: tb.oof(d, b, s) for s in tb.SEEDS})
    tb.log('OOF listo:', b)
tb.tabla(R)

[     1s] OOF listo: CNB


[     1s] OOF listo: LR


[     1s] OOF listo: SVM


[     1s] OOF listo: NB-LR


[     1s] OOF listo: Powerset


[     1s] OOF listo: LR-charc


[     1s] OOF listo: NB-LR-charc


[     1s] OOF listo: LGBM


[     1s] OOF listo: Cadena


,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
CNB,0.7509,0.7794,0.7086,0.7463,0.0023,0.7090,0.749 / 0.744 / 0.746
LR,0.7924,0.8246,0.7899,0.8023,0.0020,0.7499,0.803 / 0.805 / 0.800
SVM,0.7890,0.8279,0.7882,0.8017,0.0009,0.7527,0.803 / 0.801 / 0.801
NB-LR,0.7916,0.8231,0.7883,0.8010,0.0009,0.7578,0.801 / 0.802 / 0.800
Powerset,0.7937,0.8284,0.7889,0.8037,0.0021,0.7793,0.806 / 0.804 / 0.801
LR-charc,0.7914,0.8278,0.7921,0.8038,0.0033,0.7527,0.799 / 0.807 / 0.804
NB-LR-charc,0.7903,0.8214,0.7863,0.7993,0.0017,0.7549,0.799 / 0.797 / 0.801
LGBM,0.7858,0.8182,0.7743,0.7928,0.0007,0.7410,0.792 / 0.793 / 0.793
Cadena,0.7924,0.8168,0.7868,0.7987,0.0014,0.7747,0.798 / 0.801 / 0.797


## 2. ¿Aportan diversidad? Correlación de las probabilidades OOF de HS (semilla 42)
Si un modelo nuevo está muy correlacionado con los originales (> 0,95), aporta poco al stacking.

In [3]:
pd.DataFrame({b: tb.oof(d, b, 42)[:, 0] for b in tb.BASES_ORIGINALES + NUEVOS}).corr().round(3)

,CNB,LR,SVM,NB-LR,Powerset,LR-charc,NB-LR-charc,LGBM,Cadena
CNB,1.000,0.844,0.846,0.869,0.836,0.836,0.863,0.839,0.844
LR,0.844,1.000,0.993,0.965,0.977,0.995,0.965,0.910,1.000
SVM,0.846,0.993,1.000,0.964,0.974,0.991,0.965,0.921,0.993
NB-LR,0.869,0.965,0.964,1.000,0.948,0.963,0.996,0.897,0.965
Powerset,0.836,0.977,0.974,0.948,1.000,0.975,0.950,0.901,0.977
LR-charc,0.836,0.995,0.991,0.963,0.975,1.000,0.969,0.912,0.995
NB-LR-charc,0.863,0.965,0.965,0.996,0.950,0.969,1.000,0.899,0.965
LGBM,0.839,0.910,0.921,0.897,0.901,0.912,0.899,1.000,0.910
Cadena,0.844,1.000,0.993,0.965,0.977,0.995,0.965,0.910,1.000


## 3. Stacking con distintos conjuntos de modelos base
Conjuntos definidos de antemano (no búsqueda exhaustiva, para no sobreajustar la selección).

In [4]:
CONJUNTOS = {
    'E0 · 5 originales': tb.BASES_ORIGINALES,
    '5 originales + LR-charc + NB-LR-charc': tb.BASES_ORIGINALES + ['LR-charc', 'NB-LR-charc'],
    '5 originales + LGBM': tb.BASES_ORIGINALES + ['LGBM'],
    '5 originales + Cadena': tb.BASES_ORIGINALES + ['Cadena'],
    'Los 9 modelos': tb.BASES_ORIGINALES + NUEVOS,
    'LR + Powerset + NB-LR-charc + LGBM': ['LR', 'Powerset', 'NB-LR-charc', 'LGBM'],
}
RS = {k: tb.evaluar(d, {s: tb.stacking(d, bs, s, con_test=False)[0] for s in tb.SEEDS}) for k, bs in CONJUNTOS.items()}
tb.tabla(RS)

,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
E0 · 5 originales,0.8038,0.8345,0.7994,0.8126,0.0021,0.7757,0.815 / 0.810 / 0.813
5 originales + LR-charc + NB-LR-charc,0.8046,0.8336,0.7978,0.8120,0.0018,0.7749,0.813 / 0.810 / 0.813
5 originales + LGBM,0.8072,0.8336,0.7982,0.8130,0.0013,0.7737,0.814 / 0.813 / 0.811
5 originales + Cadena,0.8021,0.8319,0.7989,0.8110,0.0023,0.7747,0.814 / 0.808 / 0.811
Los 9 modelos,0.8038,0.8333,0.7981,0.8117,0.0015,0.7732,0.810 / 0.813 / 0.813
LR + Powerset + NB-LR-charc + LGBM,0.8063,0.8322,0.7996,0.8127,0.0007,0.7739,0.812 / 0.812 / 0.814


## Conclusión: **no se adopta**
- Ningún modelo nuevo supera a los originales por separado. LR-charc (0,804) empata con Powerset y LGBM es el más débil (0,793).
- En el stacking, todas las variantes quedan entre 0,811 y 0,813, **dentro de la desviación entre semillas (±0,002)**. La mejor (+LGBM, 0,8130) gana apenas 0,0004 y no cumple el criterio de ≥ 0,003.
- LGBM es el modelo menos correlacionado (el único no lineal) y por eso es el único que mueve algo. Aun así, como todos ven las mismas palabras, comparten la mayoría de los errores.
- Lectura para el informe: el stacking está **saturado** con estas representaciones. Las ganancias tendrían que venir de **información nueva** (léxicos externos o representaciones distribuidas), no de más clasificadores sobre los mismos n-gramas. Eso motiva E4 y E5.
- No se genera submission.